<a href="https://colab.research.google.com/github/crisriverar/Analisys-RappiPlus-/blob/main/Analisis_RappiPlus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

In [ ]:
# importar librerías
import pandas as pd
import seaborn as sns

In [ ]:
# cargar archivos
orders =  pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# explorar datasets
print("Filas y columnas:", orders.shape)
orders.head()

Filas y columnas: (24996, 12)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
print("Filas y columnas:", catalog.shape)
catalog.head(7)

Filas y columnas: (7, 4)


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [ ]:
print("Filas y columnas:", marketing.shape)
marketing.head()

Filas y columnas: (1620, 5)


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

In [ ]:
# inspección de orders con .info()
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [ ]:
# inspección y tratamiento de nulos
print(orders.isna().sum())
print(orders.isna().mean().sort_values(ascending=False))

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64
pais                  0.011952
categoria_producto    0.003187
cantidad              0.001992
precio_unitario       0.001992
monto_descuento       0.001992
fuente_referencia     0.001195
nombre_producto       0.001195
dispositivo           0.000797
id_pedido             0.000000
id_usuario            0.000000
fecha_hora_pedido     0.000000
monto_total           0.000000
dtype: float64


In [ ]:
# las columans numéricas se imputan con la mediana
orders_mediana_cantidad = orders["cantidad"].median()
orders_mediana_precio = orders["precio_unitario"].median()
orders["cantidad"] = orders["cantidad"].fillna(orders_mediana_cantidad)
orders["precio_unitario"] = orders["precio_unitario"].fillna(orders_mediana_precio)
orders["monto_descuento"] = orders["monto_descuento"].fillna(0)

In [ ]:
# las columnas categoricas nulas se remplazan con "Desconocidos" para no perder la info
columnas_categoricas = ["pais", "dispositivo", "fuente_referencia", "nombre_producto", "categoria_producto"]
orders[columnas_categoricas] = orders[columnas_categoricas].fillna("Desconocido")

In [ ]:
# inspección de valores negativos
orders[["monto_total","cantidad","precio_unitario"]].describe()

,monto_total,cantidad,precio_unitario
count,2.510000e+04,25100.000000,25100.000000
mean,2.072680e+03,7.082590,259.304373
std,9.894995e+04,295.981834,138.588215
min,-4.926500e+02,-2.000000,20.030000
25%,1.805075e+02,1.000000,138.592500
50%,3.417500e+02,2.000000,258.715000
75%,5.185800e+02,2.000000,380.092500
max,8.840200e+06,20000.000000,499.960000


In [ ]:
Q1 = orders['monto_total'].quantile(0.25)
print('Primer cuartil: ', Q1)
Q3 = orders['monto_total'].quantile(0.75)
print('Tercer cuartil: ', Q3)
IQR = Q3 - Q1
print('IQR: ', IQR)

Primer cuartil:  180.5075
Tercer cuartil:  518.58
IQR:  338.07250000000005


In [ ]:
#calcular límite inferior
lower = Q1 - 1.5 * IQR
print('Límite inferior: ', lower)
#calcular límite superior
upper = Q3 + 1.5 * IQR
print('Límite superior: ', upper)

Límite inferior:  -326.6012500000001
Límite superior:  1025.6887500000003


In [ ]:
outliers = orders[(orders['monto_total'] < lower) | (orders['monto_total'] > upper)]
print('Cantidad de outliers: ', len(outliers))
print('Porcentaje: ', round(len(outliers)/len(orders)*100, 2), '%')

Cantidad de outliers:  12
Porcentaje:  0.05 %


In [ ]:
#Se eliminan los datos negativos y los precios altos se mantienen
orders = orders[orders['cantidad'] >= 0]
print('Filas originales: ', len(orders))
print('Filas después de eliminar: ', len(orders))
print('Filas eliminadas: ', len(orders) - len(orders))
orders = orders.reset_index(drop=True)

Filas originales:  25096
Filas después de eliminar:  25096
Filas eliminadas:  0


In [ ]:
orders[["monto_total","cantidad","precio_unitario"]].describe()

,monto_total,cantidad,precio_unitario
count,2.509600e+04,25096.000000,25096.000000
mean,2.073056e+03,7.083918,259.303226
std,9.895783e+04,296.005404,138.576931
min,5.240000e+00,1.000000,20.030000
25%,1.805925e+02,1.000000,138.607500
50%,3.418050e+02,2.000000,258.715000
75%,5.185950e+02,2.000000,380.075000
max,8.840200e+06,20000.000000,499.960000


In [ ]:
# inspección de columnas categoricas
orders[["pais", "dispositivo"]].value_counts()

pais         dispositivo
Colombia     desktop        3985
Mexico       mobile         3786
             desktop        3708
Argentina    desktop        3657
             mobile         3631
Colombia     mobile         3527
mexico       desktop         438
             mobile          427
colombia     desktop         414
argentina    desktop         411
colombia     mobile          408
argentina    mobile          388
Desconocido  mobile          154
             desktop         142
Mexico       Desconocido       8
Colombia     Desconocido       8
Argentina    Desconocido       3
colombia     Desconocido       1
dtype: int64

In [ ]:
# Corrección de los nombres de los paises
orders['pais'] = orders['pais'].str.strip().str.title()

In [ ]:
# inspección de duplicados
print("Cantidad de filas duplicadas:",orders.duplicated().sum())

Cantidad de filas duplicadas: 100


In [ ]:
orders = orders.drop_duplicates()
print("Cantidad de filas duplicadas:",orders.duplicated().sum())

Cantidad de filas duplicadas: 0


In [ ]:
# cambio de valores a fechas
orders['fecha_hora_pedido']=pd.to_datetime(orders['fecha_hora_pedido'],errors="coerce")

In [ ]:
# inspección de catalog con .info()
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [ ]:
# inspección de marketing con .info()
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [ ]:
# inspección y tramatiento de nulos
print(marketing.isna().sum())
print(marketing.isna().mean().sort_values(ascending=False))

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64
canal         0.062346
fecha         0.000000
pais          0.000000
id_campaña    0.000000
gasto         0.000000
dtype: float64


In [ ]:
print(marketing[marketing['canal'].isna()]['pais'].value_counts())
print(marketing[marketing['canal'].isna()]['id_campaña'].value_counts())

Argentina    34
Mexico       34
Colombia     33
Name: pais, dtype: int64
social_Argentina         12
organic_Mexico           12
organic_Colombia         11
paid_search_Argentina    11
social_Mexico            11
social_Colombia          11
organic_Argentina        11
paid_search_Mexico       11
paid_search_Colombia     11
Name: id_campaña, dtype: int64


In [ ]:
# las columnas nulas se remplazan con "Desconocidos" para no perder la info
marketing['canal'] = marketing['canal'].fillna('Desconocido')
print(marketing.isna().sum())

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64


In [ ]:
# inspección de valores negativos
marketing["gasto"].describe()

count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64

In [ ]:
# inspección de columnas categoricas
marketing[["pais", "id_campaña","canal"]].value_counts()

pais       id_campaña             canal      
Colombia   paid_search_Colombia   paid_search    169
           organic_Colombia       organic        169
Mexico     paid_search_Mexico     paid_search    169
Colombia   social_Colombia        social         169
Argentina  organic_Argentina      organic        169
Mexico     social_Mexico          social         169
Argentina  paid_search_Argentina  paid_search    169
           social_Argentina       social         168
Mexico     organic_Mexico         organic        168
Argentina  social_Argentina       Desconocido     12
Mexico     organic_Mexico         Desconocido     12
Colombia   organic_Colombia       Desconocido     11
           paid_search_Colombia   Desconocido     11
           social_Colombia        Desconocido     11
Mexico     paid_search_Mexico     Desconocido     11
Argentina  paid_search_Argentina  Desconocido     11
Mexico     social_Mexico          Desconocido     11
Argentina  organic_Argentina      Desconocido     11


In [ ]:
# # inspección de duplicados
print("Cantidad de filas duplicadas:",marketing.duplicated().sum())

Cantidad de filas duplicadas: 0


In [ ]:
# cambio de valores a fechas
marketing['fecha']=pd.to_datetime(marketing['fecha'],errors="coerce")

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

In [ ]:
df_union = pd.merge(orders, catalog, on="nombre_producto", how="left")

In [ ]:
revenue= orders["monto_total"].sum()
print("Ingreso Total:", revenue)

Ingreso Total: 51988609.559999995


In [ ]:
df_union["costo total"]= df_union["costo_unitario"]*df_union["cantidad"]
costo_total= df_union["costo total"].sum()
print("Costo Total:", costo_total)

Costo Total: 43132326.489999995


In [ ]:
gasto_marketing= marketing["gasto"].sum()
print("Gasto Marketing:", round(gasto_marketing,2))

Gasto Marketing: 2871843.53


In [ ]:
gasto_marketing_canal= marketing.groupby("canal")["gasto"].sum()
print("Gasto de Marketing por Canal:", gasto_marketing_canal)

Gasto de Marketing por Canal: canal
Desconocido    177179.10
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64


In [ ]:
ganancia_bruta = revenue - costo_total
print("Ganancia Bruta:", ganancia_bruta)

Ganancia Bruta: 8856283.07


In [ ]:
ganancia_neta = ganancia_bruta - gasto_marketing
print("Ganancia Neta:", ganancia_neta)

Ganancia Neta: 5984439.540000001


In [ ]:
ticket_promedio= revenue / orders["monto_total"].count()
print("Ticket Promedio:", round(ticket_promedio,2))

Ticket Promedio: 2079.88


In [ ]:
productos_por_orden= orders["cantidad"].mean()
print("Promedio de Productos por Orden: ", round(productos_por_orden, 2))

Promedio de Productos por Orden:  7.11


In [ ]:
producto_mas_vendido = orders["nombre_producto"].value_counts().idxmax()
print("Producto más vendido:", producto_mas_vendido)

Producto más vendido: Blender-XL-Red


In [ ]:
margen_de_ganancia= ganancia_neta/revenue*100
print("Margen de Ganancia:", round(margen_de_ganancia,2))

Margen de Ganancia: 11.51


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


In [ ]:

import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''

WITH
cte_first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'

),
cte_select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'

),
cte_add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento  = 'add_to_cart'
),
cte_begin_checkout AS (

    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'

),
cte_add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),

cte_purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
)
SELECT

    (SELECT COUNT(*) FROM cte_first_visit) AS first_visit_usuarios,
    (SELECT COUNT(*) FROM cte_select_item ) AS select_item_usuarios,
    (SELECT COUNT(*) FROM cte_add_to_cart) AS add_to_cart_usuarios,
    (SELECT COUNT(*) FROM cte_begin_checkout) AS begin_checkout_usuarios,
    (SELECT COUNT(*) FROM cte_add_payment_info) add_payment_info_usuarios,
    (SELECT COUNT(*) FROM cte_purchase) AS purchase_usuarios


'''

totals = pd.read_sql(query_totals, con=engine)
totals

,first_visit_usuarios,select_item_usuarios,add_to_cart_usuarios,begin_checkout_usuarios,add_payment_info_usuarios,purchase_usuarios
0,7796,7582,7634,7208,6250,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''

WITH
cte_first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'

),
cte_select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'

),
cte_add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento  = 'add_to_cart'
),
cte_begin_checkout AS (

    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'

),
cte_add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),

cte_purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
)
SELECT
    (SELECT COUNT(*) FROM cte_first_visit) AS first_visit_usuarios,
    (SELECT COUNT(*) FROM cte_select_item ) AS select_item_usuarios,
    (SELECT COUNT(*) FROM cte_add_to_cart) AS add_to_cart_usuarios,
    (SELECT COUNT(*) FROM cte_begin_checkout) AS begin_checkout_usuarios,
    (SELECT COUNT(*) FROM cte_add_payment_info) add_payment_info_usuarios,
    (SELECT COUNT(*) FROM cte_purchase) AS purchase_usuarios,

    ((SELECT COUNT(*) FROM cte_first_visit) - (SELECT COUNT(*) FROM cte_select_item)) * 100.0
        / NULLIF((SELECT COUNT(*) FROM cte_first_visit), 0) AS dropoff_after_first_visit_pct,

    ((SELECT COUNT(*) FROM cte_select_item) - (SELECT COUNT(*) FROM cte_add_to_cart)) * 100.0
        / NULLIF((SELECT COUNT(*) FROM cte_select_item), 0) AS dropoff_after_select_item_pct,

    ((SELECT COUNT(*) FROM cte_add_to_cart) - (SELECT COUNT(*) FROM cte_begin_checkout)) * 100.0
        / NULLIF((SELECT COUNT(*) FROM cte_add_to_cart), 0) AS dropoff_after_add_to_cart_pct,

    ((SELECT COUNT(*) FROM cte_begin_checkout) - (SELECT COUNT(*) FROM cte_add_payment_info)) * 100.0
        / NULLIF((SELECT COUNT(*) FROM cte_begin_checkout), 0) AS dropoff_after_begin_checkout_pct,

    ((SELECT COUNT(*) FROM cte_add_payment_info) - (SELECT COUNT(*) FROM cte_purchase)) * 100.0
        / NULLIF((SELECT COUNT(*) FROM cte_add_payment_info), 0) AS dropoff_after_add_payment_pct;

'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_visit_usuarios,select_item_usuarios,add_to_cart_usuarios,begin_checkout_usuarios,add_payment_info_usuarios,purchase_usuarios,dropoff_after_first_visit_pct,dropoff_after_select_item_pct,dropoff_after_add_to_cart_pct,dropoff_after_begin_checkout_pct,dropoff_after_add_payment_pct
0,7796,7582,7634,7208,6250,6240,2.744997,-0.685835,5.580299,13.290788,0.16


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

In [ ]:

# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''

WITH cte_cohort AS (
    SELECT
        id_usuario,
        TO_CHAR(DATE_TRUNC('month', fecha_registro::date), 'YYYY-MM') AS cohort
    FROM users
),
    cte_totales AS (
    SELECT
        cohort,
        COUNT(DISTINCT id_usuario) AS total_usuarios
    FROM cte_cohort
    GROUP BY cohort
),
cte_activos AS (
    SELECT
        c.cohort,
        COUNT(DISTINCT CASE WHEN FLOOR(a.dias_despues_registro / 7) = 1 THEN a.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN FLOOR(a.dias_despues_registro / 7) = 2 THEN a.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN FLOOR(a.dias_despues_registro / 7) = 3 THEN a.id_usuario END) AS retenido_w3
    FROM user_activity AS a
    INNER JOIN cte_cohort AS c
        ON a.id_usuario = c.id_usuario
    WHERE a.activo = 1
    GROUP BY c.cohort
)

SELECT
    t.cohort,
    t.total_usuarios,
    ROUND(COALESCE(a.retenido_w1, 0) / (t.total_usuarios *1.0)*100,2) AS semana_1,
    ROUND(COALESCE(a.retenido_w2, 0) / (t.total_usuarios *1.0)*100,2) AS semana_2,
    ROUND(COALESCE(a.retenido_w3, 0) / (t.total_usuarios *1.0)*100,2) AS semana_3
    FROM cte_totales AS t
    LEFT JOIN cte_activos AS a
    ON t.cohort = a.cohort
    ORDER BY t.cohort;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohort,total_usuarios,semana_1,semana_2,semana_3
0,2025-01,1627,42.84,41.06,40.32
1,2025-02,1444,42.31,42.17,43.98
2,2025-03,1636,41.38,43.09,42.18
3,2025-04,1606,42.34,43.40,41.28
4,2025-05,1687,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):**La tasa de conversión es igual en ambas páginas. Cualquier diferencia se debe al azar del muestreo.
   - **H₁ (Hipótesis alternativa):**  La tasa de conversión es diferente entre ambas páginas. Hay un factor que influye en la decisión de compra.
   
**Test estadístico:** z-test  
**Nivel de significancia alpha:** 0.05

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
experiment =  pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
# Número de usuarios convertidos por página
conversiones = experiment.groupby("variante")["convirtio"].sum()

# Total de usuarios por página
visitas = experiment.groupby("variante")["convirtio"].count()

print("Conversiones:", conversiones)
print("\nVisitas:", visitas)

Conversiones: variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

Visitas: variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64


In [ ]:
exitos = [conversiones['control'], conversiones['tratamiento']]
observaciones = [visitas['control'], visitas['tratamiento']]

In [ ]:
z_stat, p_value = proportions_ztest(exitos , observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")

Estadístico z: -0.8132782986429474
Valor p: 0.41605851639119995


In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")

No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.


In [ ]:
tasa_A = exitos[0] / observaciones[0]
tasa_B = exitos[1] / observaciones[1]

if tasa_A > tasa_B:
    print(f"\nLa versión A tiene una mayor tasa de conversión ({tasa_A - tasa_B:.2%}).")
elif tasa_B > tasa_A:
    print(f"\nLa versión B tiene una mayor tasa de conversión ({tasa_B - tasa_A:.2%})")
else:
    print("\nAmbas páginas tienen la misma tasa de conversión.")


La versión B tiene una mayor tasa de conversión (0.60%)
